In [1]:
import pandas as pd
import numpy as np
import os
import json
import requests

from pyjstat import pyjstat
from collections import OrderedDict

In [2]:
dir_out = "../parsed_data/"

In [3]:
map_country_ISO = {
    "Belgium" : "BE",
    "Bulgaria" : "BG",
    "Czechia" : "CZ",
    "Denmark" : "DK",
    "Germany (until 1990 former territory of the FRG)" : "DE",
    "Estonia" : "EE",
    "Ireland" : "IE",
    "Greece" : "GR",
    "Spain" : "ES",
    "France" : "FR",
    "Croatia" : "HR",
    "Italy" : "IT",
    "Cyprus" : "CY",
    "Latvia" : "LV",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Hungary" : "HU",
    "Malta" : "MT",
    "Netherlands" : "NL",
    "Austria" : "AT",
    "Poland" : "PL",
    "Portugal" : "PT",
    "Romania" : "RO",
    "Slovenia" : "SI",
    "Slovakia" : "SK",
    "Finland" : "FI",
    "Sweden" : "SE",
    "United Kingdom" : "GB",
    "Iceland" : "IS",
    "Norway" : "NO",
    "Montenegro" : "ME",
    "North Macedonia" : "MK",
    "Albania" : "AL",
    "Serbia" : "RS",
    "Turkey" : "TR",
    "Bosnia and Herzegovina" : "BA",
    "Kosovo (under United Nations Security Council Resolution 1244/99)" : "XK",
    "Moldova" : "MD",
    "Ukraine" : "UA",
    "Georgia" : "GE"
}

In [4]:
map_year_vintages ={
    '2017' : 'Less than 2 years',
    '2016' : 'Less than 2 years',
    '2015' : 'Less than 2 years',
    '2014' : 'From 2 to 5 years',
    '2013' : 'From 2 to 5 years',
    '2012' : 'From 2 to 5 years',
    '2011' : 'From 5 to 10 years',
    '2010' : 'From 5 to 10 years',
    '2009' : 'From 5 to 10 years',
    '2008' : 'From 5 to 10 years',
    '2007' : 'From 5 to 10 years',
    '2006' : 'From 10 to 20 years',
    '2005' : 'From 10 to 20 years',
    '2004' : 'From 10 to 20 years',
    '2003' : 'From 10 to 20 years',
    '2002' : 'From 10 to 20 years',
    '2001' : 'From 10 to 20 years',
    '2000' : 'From 10 to 20 years'
}

In [5]:
vintages = ['From 10 to 20 years', 'From 2 to 5 years', 'From 5 to 10 years',
           'Less than 2 years', 'Over 20 years']

Download and parse Eurostat data on number of cars by age

In [6]:
# Eurostat queries based on query builder: 
# https://ec.europa.eu/eurostat/web/json-and-unicode-web-services/getting-started/query-builder
indicator = 'road_eqs_carage'
dataformat = 'json'

params = dict(
    precision = 1,
    time=2017,
    geo = {'AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MD', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UA', 'UK', 'AL', 'BA', 'LI', 'IS', 'GE', 'ME', 'XK'},
)

In [7]:
url = 'http://ec.europa.eu/eurostat/wdds/rest/data/v2.1/'+dataformat+'/en/'+indicator+'?'
r = requests.get(url=url, params=params)
road_eqs_carage = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])

In [8]:
road_eqs_carage = road_eqs_carage.rename(columns={'geo':'country'})
road_eqs_carage = road_eqs_carage.rename(columns={'value':'cars'})
road_eqs_carage['country'] = road_eqs_carage['country'].map(map_country_ISO)
road_eqs_carage = road_eqs_carage.groupby(['country','age']).sum()

In [9]:
road_eqs_carage.loc['BG','From 10 to 20 years'] = road_eqs_carage.loc['BG','Total']
road_eqs_carage.loc['GR','From 10 to 20 years'] = road_eqs_carage.loc['GR','Total']
road_eqs_carage.loc['SK','From 10 to 20 years'] = road_eqs_carage.loc['SK','Total']
road_eqs_carage.loc['IT','From 10 to 20 years'] = 38520000
road_eqs_carage.loc['IS','From 10 to 20 years'] = 257100

In [10]:
road_eqs_carage_vintages = road_eqs_carage[road_eqs_carage.index.get_level_values('age').isin(vintages)]
road_eqs_carage_vintages.head()

cars
country age                           
AT      From 10 to 20 years  1405215.0
        From 2 to 5 years     890343.0
        From 5 to 10 years   1432456.0
        Less than 2 years     889536.0
        Over 20 years         281028.0

Download and parse Eurostat data on average emissions by car age

In [11]:
# Eurostat queries based on query builder: 
# https://ec.europa.eu/eurostat/web/json-and-unicode-web-services/getting-started/query-builder
indicator = 't2020_rk330'
dataformat = 'json'

params = dict(
    sinceTimePeriod = '2000',
    precision = 1,
    geo = {'AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MD', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UA', 'UK', 'AL', 'BA', 'LI', 'IS', 'GE', 'ME', 'XK'},
)

In [12]:
url = 'http://ec.europa.eu/eurostat/wdds/rest/data/v2.1/'+dataformat+'/en/'+indicator+'?'
r = requests.get(url=url, params=params)
t2020_rk330 = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])

In [13]:
#rename countries and convert to MWh
t2020_rk330 = t2020_rk330.dropna()
t2020_rk330 = t2020_rk330.rename(columns={'geo':'country'})
t2020_rk330 = t2020_rk330.rename(columns={'value':'gperkm'})
t2020_rk330['country'] = t2020_rk330['country'].map(map_country_ISO)
t2020_rk330['age'] = t2020_rk330['time'].map(map_year_vintages)
t2020_rk330.head()

INFO:numexpr.utils:NumExpr defaulting to 8 threads.


,country,time,gperkm,age
0,AT,2000,168.0,From 10 to 20 years
1,AT,2001,165.6,From 10 to 20 years
2,AT,2002,164.4,From 10 to 20 years
3,AT,2003,163.8,From 10 to 20 years
4,AT,2004,161.9,From 10 to 20 years


In [14]:
t2020_rk330 = t2020_rk330.groupby(['age','country']).mean()
t2020_rk330.head()

gperkm
age                 country            
From 10 to 20 years AT       164.214286
                    BE       159.285714
                    CY       172.166667
                    CZ       154.500000
                    DE       176.542857

In [15]:
#we fill missing values with maximum and roughly interpolate over 20 years value
t2020_rk330 = t2020_rk330.reset_index().pivot_table(index='country',columns='age',values='gperkm')
t2020_rk330['From 5 to 10 years'] = t2020_rk330['From 5 to 10 years'].fillna(t2020_rk330['From 5 to 10 years'].max())
t2020_rk330['From 10 to 20 years'] = t2020_rk330['From 10 to 20 years'].fillna(t2020_rk330['From 10 to 20 years'].max())
t2020_rk330['Over 20 years'] = t2020_rk330['From 10 to 20 years'] + t2020_rk330['From 10 to 20 years'] - t2020_rk330['From 5 to 10 years']
t2020_rk330 = pd.DataFrame(t2020_rk330.stack()).rename(columns={0:'gperkm'})
t2020_rk330.head()

gperkm
country age                            
AT      From 10 to 20 years  164.214286
        From 2 to 5 years    131.933333
        From 5 to 10 years   150.780000
        Less than 2 years    121.600000
        Over 20 years        177.648571

In [16]:
df_passengervehicles = road_eqs_carage_vintages.join(t2020_rk330,on=['country','age'])
df_passengervehicles['product'] = df_passengervehicles['cars'] * df_passengervehicles['gperkm']
df_passengervehicles.head()

cars      gperkm       product
country age                                                     
AT      From 10 to 20 years  1405215.0  164.214286  2.307564e+08
        From 2 to 5 years     890343.0  131.933333  1.174659e+08
        From 5 to 10 years   1432456.0  150.780000  2.159857e+08
        Less than 2 years     889536.0  121.600000  1.081676e+08
        Over 20 years         281028.0  177.648571  4.992422e+07

In [17]:
df_passengervehicles_total =  df_passengervehicles.reset_index().groupby('country').sum().drop(columns={'gperkm'}).rename(columns={'cars':'cars_total'})
df_passengervehicles_total['gperkm'] = df_passengervehicles_total['product'] / df_passengervehicles_total['cars_total']
df_passengervehicles_total = df_passengervehicles_total.drop(columns={'product'})
df_passengervehicles_total.head()

,cars_total,gperkm
country,,
AT,4898578.0,147.450916
BE,5785447.0,138.450122
BG,2770615.0,196.642857
CY,526617.0,166.144999
CZ,5538222.0,148.749391


Download and parse Eurostat data on number of cars by fuel type

In [18]:
# Eurostat queries based on query builder: 
# https://ec.europa.eu/eurostat/web/json-and-unicode-web-services/getting-started/query-builder
indicator = 'road_eqs_carpda'
dataformat = 'json'

params = dict(
    time=2017,
    precision = 1,
    geo = {'AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MD', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UA', 'UK', 'AL', 'BA', 'LI', 'IS', 'GE', 'ME', 'XK'},
)

In [19]:
url = 'http://ec.europa.eu/eurostat/wdds/rest/data/v2.1/'+dataformat+'/en/'+indicator+'?'
r = requests.get(url=url, params=params)
road_eqs_carpda = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])

In [20]:
road_eqs_carpda['country'] = road_eqs_carpda['geo'].map(map_country_ISO)
road_eqs_carpda = road_eqs_carpda[['country','value','mot_nrg']]
road_eqs_carpda = road_eqs_carpda[road_eqs_carpda.mot_nrg.isin(['Electricity','Total'])]
road_eqs_carpda = road_eqs_carpda.pivot_table(index='country',columns='mot_nrg',values='value')
road_eqs_carpda['EV_share'] = road_eqs_carpda['Electricity'] / road_eqs_carpda['Total']
road_eqs_carpda = pd.DataFrame(road_eqs_carpda['EV_share'])
road_eqs_carpda = road_eqs_carpda.fillna(0)
road_eqs_carpda.head()

,EV_share
country,
AT,0.002984
BE,0.001478
BG,0.000000
CY,0.000165
CZ,0.000275


In [21]:
df_passengervehicles_total = df_passengervehicles_total.join(road_eqs_carpda, on='country')

In [22]:
df_passengervehicles_total.loc['IT']['EV_share'] = 7975/df_passengervehicles_total.loc['IT']['cars_total']

In [23]:
df_passengervehicles_total.to_csv(dir_out+'passengervehicles_Eurostat.csv', encoding="utf-8")